# Play Store Data - Interactive Choropleth Map Analysis



In [1]:
!pip install pandas plotly numpy pytz -q

In [ ]:
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import numpy as np
from datetime import datetime
import pytz


from google.colab import files
uploaded = files.upload()

Saving Play Store Data.csv to Play Store Data.csv


In [ ]:
df = pd.read_csv('Play Store Data.csv')
print("Dataset shape:", df.shape)
print("\nFirst few rows:")
print(df.head(5))
print("\nColumns:")
df.columns.tolist()

Dataset shape: (10841, 13)

First few rows:
                                                 App        Category  Rating  \
0     Photo Editor & Candy Camera & Grid & ScrapBook  ART_AND_DESIGN     4.1   
1                                Coloring book moana  ART_AND_DESIGN     3.9   
2  U Launcher Lite – FREE Live Cool Themes, Hide ...  ART_AND_DESIGN     4.7   
3                              Sketch - Draw & Paint  ART_AND_DESIGN     4.5   
4              Pixel Draw - Number Art Coloring Book  ART_AND_DESIGN     4.3   

  Reviews  Size     Installs  Type Price Content Rating  \
0     159   19M      10,000+  Free     0       Everyone   
1     967   14M     500,000+  Free     0       Everyone   
2   87510  8.7M   5,000,000+  Free     0       Everyone   
3  215644   25M  50,000,000+  Free     0           Teen   
4     967  2.8M     100,000+  Free     0       Everyone   

                      Genres      Last Updated         Current Ver  \
0               Art & Design   January 7, 2018    

['App',
 'Category',
 'Rating',
 'Reviews',
 'Size',
 'Installs',
 'Type',
 'Price',
 'Content Rating',
 'Genres',
 'Last Updated',
 'Current Ver',
 'Android Ver']

In [ ]:
def convert_installs_to_numeric(installs_str):
    if pd.isna(installs_str) or installs_str == 'Free' or installs_str == '0':
        return 0
    installs_str = str(installs_str).replace(',', '').replace('+', '')
    try:
        return float(installs_str)
    except:
        return 0

df['Installs_Numeric'] = df['Installs'].apply(convert_installs_to_numeric)
print("Sample conversions:")
print(df[['Installs', 'Installs_Numeric']].head(10))

Sample conversions:
      Installs  Installs_Numeric
0      10,000+           10000.0
1     500,000+          500000.0
2   5,000,000+         5000000.0
3  50,000,000+        50000000.0
4     100,000+          100000.0
5      50,000+           50000.0
6      50,000+           50000.0
7   1,000,000+         1000000.0
8   1,000,000+         1000000.0
9      10,000+           10000.0


In [ ]:
# Filter 1: Exclude categories starting with A, C, G, S
excluded_letters = ['A', 'C', 'G', 'S']
df_filtered = df[~df['Category'].str.startswith(tuple(excluded_letters))].copy()

print(f"Dataset after filtering: {len(df_filtered)} rows")
print(f"Rows removed: {len(df) - len(df_filtered)}")
print("\\nUnique categories (after filter):")
print(sorted(df_filtered['Category'].unique()))

Dataset after filtering: 8161 rows
Rows removed: 2680
\nUnique categories (after filter):
['1.9', 'BEAUTY', 'BOOKS_AND_REFERENCE', 'BUSINESS', 'DATING', 'EDUCATION', 'ENTERTAINMENT', 'EVENTS', 'FAMILY', 'FINANCE', 'FOOD_AND_DRINK', 'HEALTH_AND_FITNESS', 'HOUSE_AND_HOME', 'LIBRARIES_AND_DEMO', 'LIFESTYLE', 'MAPS_AND_NAVIGATION', 'MEDICAL', 'NEWS_AND_MAGAZINES', 'PARENTING', 'PERSONALIZATION', 'PHOTOGRAPHY', 'PRODUCTIVITY', 'TOOLS', 'TRAVEL_AND_LOCAL', 'VIDEO_PLAYERS', 'WEATHER']


In [ ]:
# Get top 5 categories by total installs
category_installs = df_filtered.groupby('Category')['Installs_Numeric'].sum().sort_values(ascending=False)
print("Top 10 categories by total installs:")
print(category_installs.head(10))

# Filter for top 5 categories
top_5_categories = category_installs.head(5).index.tolist()
print(f"\nTop 5 categories: {top_5_categories}")

df_top5 = df_filtered[df_filtered['Category'].isin(top_5_categories)].copy()
print(f"Total apps in top 5 categories: {len(df_top5)}")

Top 10 categories by total installs:
Category
PRODUCTIVITY           1.417609e+10
TOOLS                  1.145277e+10
FAMILY                 1.025826e+10
PHOTOGRAPHY            1.008825e+10
NEWS_AND_MAGAZINES     7.496318e+09
TRAVEL_AND_LOCAL       6.868887e+09
VIDEO_PLAYERS          6.222003e+09
ENTERTAINMENT          2.869160e+09
PERSONALIZATION        2.325495e+09
BOOKS_AND_REFERENCE    1.921470e+09
Name: Installs_Numeric, dtype: float64

Top 5 categories: ['PRODUCTIVITY', 'TOOLS', 'FAMILY', 'PHOTOGRAPHY', 'NEWS_AND_MAGAZINES']
Total apps in top 5 categories: 3857


In [ ]:
# Map top 5 categories to countries for Choropleth visualization
category_to_country = {
    top_5_categories[0]: 'United States',
    top_5_categories[1]: 'India',
    top_5_categories[2]: 'China',
    top_5_categories[3]: 'Brazil',
    top_5_categories[4]: 'Japan'
}

print("Category to Country Mapping:")
for cat, country in category_to_country.items():
    total_installs = df_top5[df_top5['Category'] == cat]['Installs_Numeric'].sum()
    print(f"{cat}: {country} (Total Installs: {total_installs:,.0f})")

Category to Country Mapping:
PRODUCTIVITY: United States (Total Installs: 14,176,091,369)
TOOLS: India (Total Installs: 11,452,771,915)
FAMILY: China (Total Installs: 10,258,263,505)
PHOTOGRAPHY: Brazil (Total Installs: 10,088,247,655)
NEWS_AND_MAGAZINES: Japan (Total Installs: 7,496,317,760)


In [ ]:
# Create aggregated data for Choropleth
choropleth_data = []

for category, country in category_to_country.items():
    category_df = df_top5[df_top5['Category'] == category]
    total_installs = category_df['Installs_Numeric'].sum()
    num_apps = len(category_df)
    avg_rating = category_df['Rating'].mean()
    exceeds_1m = 'Yes' if total_installs > 1_000_000 else 'No'

    choropleth_data.append({
        'Country': country,
        'Category': category,
        'Total_Installs': total_installs,
        'Num_Apps': num_apps,
        'Avg_Rating': avg_rating,
        'Exceeds_1M': exceeds_1m
    })

choropleth_df = pd.DataFrame(choropleth_data)
print("Choropleth Data Summary:")
print(choropleth_df.to_string(index=False))

Choropleth Data Summary:
      Country           Category  Total_Installs  Num_Apps  Avg_Rating Exceeds_1M
United States       PRODUCTIVITY    1.417609e+10       424    4.211396        Yes
        India              TOOLS    1.145277e+10       843    4.047411        Yes
        China             FAMILY    1.025826e+10      1972    4.192272        Yes
       Brazil        PHOTOGRAPHY    1.008825e+10       335    4.192114        Yes
        Japan NEWS_AND_MAGAZINES    7.496318e+09       283    4.132189        Yes


In [ ]:
def is_within_display_time():
    ist = pytz.timezone('Asia/Kolkata')
    current_time_ist = datetime.now(ist)
    current_hour = current_time_ist.hour
    is_within = 18 <= current_hour < 20
    print(f"Current IST Time: {current_time_ist.strftime('%Y-%m-%d %H:%M:%S %Z')}")
    print(f"Display enabled (6 PM - 8 PM IST): {is_within}")
    return is_within

is_display_time = is_within_display_time()

Current IST Time: 2026-08-08 18:02:25 IST
Display enabled (6 PM - 8 PM IST): True


In [ ]:
if is_display_time:
    fig = go.Figure(data=go.Choropleth(
        locations=choropleth_df['Country'],
        z=choropleth_df['Total_Installs'],
        text=choropleth_df['Category'],
        customdata=np.column_stack((
            choropleth_df['Category'],
            choropleth_df['Total_Installs'].apply(lambda x: f"{x:,.0f}"),
            choropleth_df['Num_Apps'],
            choropleth_df['Avg_Rating'].round(2),
            choropleth_df['Exceeds_1M']
        )),
        hovertemplate='<b>%{customdata[0]}</b><br>Country: %{locations}<br>Total Installs: %{customdata[1]}<br>Number of Apps: %{customdata[2]}<br>Avg Rating: %{customdata[3]}<br>Exceeds 1M: %{customdata[4]}<br><extra></extra>',
        colorscale='Viridis',
        marker_line_color='darkgray',
        marker_line_width=2,
        colorbar=dict(
            title="Total Installs",
            thickness=15,
            len=0.7
        )
    ))

    fig.update_layout(
        title='Play Store Apps - Global Installs by Category (Top 5) | 6-8 PM IST Only',
        geo=dict(projection_type='natural earth'),
        width=1200,
        height=700
    )

    fig.show()
    print("Choropleth map displayed successfully!")
else:
    print("Display Restriction Active")
    print("The Choropleth map is only available between 6 PM - 8 PM IST.")
    print("\\nData Summary (Available anytime):")
    print(choropleth_df.to_string(index=False))

Choropleth map displayed successfully!


## Additional Analysis and Statistics

In [ ]:

print("DETAILED ANALYSIS - TOP 5 CATEGORIES")
print("\n")

for category in top_5_categories:
    cat_data = df_top5[df_top5['Category'] == category]
    total_installs = cat_data['Installs_Numeric'].sum()
    print(f"\nCategory: {category}")
    print(f"Total Apps: {len(cat_data)}")
    print(f"Total Installs: {total_installs:,.0f}")
    print(f"Exceeds 1M: {'YES' if total_installs > 1_000_000 else 'NO'}")
    print(f"Average Rating: {cat_data['Rating'].mean():.2f}")

DETAILED ANALYSIS - TOP 5 CATEGORIES



Category: PRODUCTIVITY
Total Apps: 424
Total Installs: 14,176,091,369
Exceeds 1M: YES
Average Rating: 4.21

Category: TOOLS
Total Apps: 843
Total Installs: 11,452,771,915
Exceeds 1M: YES
Average Rating: 4.05

Category: FAMILY
Total Apps: 1972
Total Installs: 10,258,263,505
Exceeds 1M: YES
Average Rating: 4.19

Category: PHOTOGRAPHY
Total Apps: 335
Total Installs: 10,088,247,655
Exceeds 1M: YES
Average Rating: 4.19

Category: NEWS_AND_MAGAZINES
Total Apps: 283
Total Installs: 7,496,317,760
Exceeds 1M: YES
Average Rating: 4.13


In [ ]:
# Convert required columns to numeric
df_top5['Installs_Numeric'] = pd.to_numeric(
    df_top5['Installs_Numeric'], errors='coerce'
)

df_top5['Rating'] = pd.to_numeric(
    df_top5['Rating'], errors='coerce'
)

df_top5['Reviews'] = pd.to_numeric(
    df_top5['Reviews'], errors='coerce'
)



Summary Statistics Table

In [ ]:
summary_stats = df_top5.groupby('Category').agg({
    'Installs_Numeric': ['sum', 'mean'],
    'Rating': 'mean',
    'Reviews': 'mean',
    'App': 'count'
}).round(2)

summary_stats.columns = ['Total Installs', 'Avg Installs per App', 'Avg Rating', 'Avg Reviews', 'Total Apps']
summary_stats = summary_stats.sort_values('Total Installs', ascending=False)

print("\n" + "=" * 100)
print("SUMMARY STATISTICS - TOP 5 CATEGORIES")
print("=" * 100)
print(summary_stats.to_string())

\n====================================================================================================
SUMMARY STATISTICS - TOP 5 CATEGORIES
                    Total Installs  Avg Installs per App  Avg Rating  Avg Reviews  Total Apps
Category                                                                                     
PRODUCTIVITY          1.417609e+10           33434177.76        4.21    269143.81         424
TOOLS                 1.145277e+10           13585731.81        4.05    324062.92         843
FAMILY                1.025826e+10            5201959.18        4.19    208025.52        1972
PHOTOGRAPHY           1.008825e+10           30114172.10        4.19    637363.13         335
NEWS_AND_MAGAZINES    7.496318e+09           26488755.34        4.13    192229.20         283
